# Disease ↔ ChemicalEntity Relation-Wise Merge

Merges Disease–Chemical triples from Monarch, PrimeKG (×3), PharmKG, and TARKG;
resolves disease names via DO/MESH and chemical names via PubChem/DrugBank;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd
import numpy as np
import re

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_CHEMICALENTITY/ALL_DISEASE_CHEMICALENTITY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Chemical — PubChem and DrugBank

In [2]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem
print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


In [3]:
Drugbank = pd.read_csv(DB_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))
print(f"DrugBank dict: {len(Drugbank_dict):,} entries")

DrugBank dict: 16,575 entries


### 1b. Disease — DO, MedGen, MESH

In [4]:
# Disease Ontology
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [5]:
# MedGen: source_id → preferred name; MONDO → MESH CUI
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict    = dict(zip(MONDO_2_MESH['source_id'],     MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'], MONDO_2_MESH['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [6]:
# MESH combined and supplementary: ID → Name
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

MESH dict: 38,520 | MESH supp: 324,150


## 2. Helper — Resolve Disease Head Node

In [7]:
def resolve_disease_node(df, node):
    """Map MONDO→MESH, derive detail_name, assign id_is for head or tail."""
    df[node] = df[node].map(MONDO_2_MESH_dict).fillna(df[node])
    df = df[~df[node].astype(str).str.startswith('MONDO')].copy()
    df[f'{node}_detail_name'] = df[node].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
    )
    df[f'{node}_id_is'] = np.where(
        df[node].isna(), None,
        np.where(df[node].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    return df

## 3. Load KG Sources

### 3a. Monarch

In [8]:
Monarch_Disease_Chemical = pd.read_csv(PROC_DIR + 'Monarch/Human/Disease_ChemicalEntity_Monarch.csv')
Monarch_Disease_Chemical.columns = Monarch_Disease_Chemical.columns.str.lower()
Monarch_Disease_Chemical.rename(columns={'kgsource': 'kg_source', 'head_name': 'head_detail_name', 'tail_name': 'tail_detail_name'}, inplace=True)
Monarch_Disease_Chemical = resolve_disease_node(Monarch_Disease_Chemical, 'head')
Monarch_Disease_Chemical['tail_id_is'] = 'Pubchem'
print(f"Monarch:    {len(Monarch_Disease_Chemical):,} rows")

Monarch_Disease_Chemical['kg_type'] = 'Generalised'
Monarch_Disease_Chemical['kg_source'] = 'MonarchKG'
Monarch_Disease_Chemical['species'] = 'Homo species'

Monarch_Disease_Chemical

Monarch:    20 rows


,head,tail,relation_type,relation_source,kg_source,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species,head_do_name,tail_id,kg_type,species
0,C0568062,126941,related_to,infores:mondo,MonarchKG,methotrexate response - Toxicity,Disease,MESH,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0034212,CHEBI:44185,Generalised,Homo species
4,DOID:670,3007,related_to,infores:mondo,MonarchKG,amphetamine abuse,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0003969,CHEBI:2679,Generalised,Homo species
5,DOID:809,446220,related_to,infores:mondo,MonarchKG,cocaine abuse,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0004456,CHEBI:27958,Generalised,Homo species
6,DOID:8519,3262018,related_to,infores:mondo,MonarchKG,barbiturate abuse,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0004599,CHEBI:29745,Generalised,Homo species
7,DOID:9975,446220,related_to,infores:mondo,MonarchKG,cocaine dependence,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0005186,CHEBI:27958,Generalised,Homo species
8,DOID:5062,6468,related_to,infores:mondo,MonarchKG,phencyclidine abuse,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0005912,CHEBI:8058,Generalised,Homo species
9,C1851920,6047,related_to,infores:mondo,MonarchKG,Dystonia 5,Disease,MESH,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0007495,CHEBI:15765,Generalised,Homo species
10,DOID:0060066,1054,related_to,infores:mondo,MonarchKG,pyridoxine-responsive sideroblastic anemia,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0008786,CHEBI:16709,Generalised,Homo species
11,DOID:10587,5280458,related_to,infores:mondo,MonarchKG,Krabbe disease,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0009499,CHEBI:16874,Generalised,Homo species
12,DOID:0050659,171548,related_to,infores:mondo,MonarchKG,biotin-responsive basal ganglia disease,Disease,DOID,NaN,NaN,...,NaN,NaN,Disease,ChemicalEntity,Disease_ChemicalEntity,NaN,MONDO:0011841,CHEBI:15956,Generalised,Homo species


### 3b. PrimeKG (×3)

Note: files 1 and 2 use the same source CSV — preserved as-is from original.

In [9]:
def load_primekg_dc(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.lower()
    df['head_detail_name'] = df['head'].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x))
    )
    df['head_id_is'] = np.where(
        df['head'].isna(), None,
        np.where(df['head'].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    df['tail'] = df['tail'].astype(str)
    df['tail_id_is'] = np.where(
        df['tail'].isna(), None,
        np.where(df['tail'].str.startswith('DB'), 'DRUGBANK', 'Pubchem')
    )
    return df

PrimeKG_Disease_Chemical1 = load_primekg_dc(PROC_DIR + 'PrimeKG/PrimeKG_Disease_Chemical.csv')
PrimeKG_Disease_Chemical2 = load_primekg_dc(PROC_DIR + 'PrimeKG/PrimeKG_Disease_ChemicalEntity1.csv')
PrimeKG_Disease_Chemical3 = load_primekg_dc(PROC_DIR + 'PrimeKG/PrimeKG_Disease_ChemicalEntity2.csv')

PrimeKG_Disease_Chemical1['kg_type'] = 'Generalised'
PrimeKG_Disease_Chemical1['kg_source'] = 'MonarchKG'
PrimeKG_Disease_Chemical1['species'] = 'Homo species'

PrimeKG_Disease_Chemical2['kg_type'] = 'Generalised'
PrimeKG_Disease_Chemical2['kg_source'] = 'MonarchKG'
PrimeKG_Disease_Chemical2['species'] = 'Homo species'

PrimeKG_Disease_Chemical3['kg_type'] = 'Generalised'
PrimeKG_Disease_Chemical3['kg_source'] = 'PrimeKG'
PrimeKG_Disease_Chemical3['species'] = 'Homo species'


print(f"PrimeKG 1: {len(PrimeKG_Disease_Chemical1):,} rows")
print(f"PrimeKG 2: {len(PrimeKG_Disease_Chemical2):,} rows")
print(f"PrimeKG 3: {len(PrimeKG_Disease_Chemical3):,} rows")

PrimeKG 1: 1,377 rows
PrimeKG 2: 1,369 rows
PrimeKG 3: 5,528 rows


In [10]:
PrimeKG_Disease_Chemical1

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,head_detail_name,kg_type,species
0,DOID:332,Disease_ChemicalEntity,6278.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,"1,1,1-trichloroethane","1,1,1-trichloroethane",CC(Cl)(Cl)Cl,CC(Cl)(Cl)Cl,amyotrophic lateral sclerosis,Generalised,Homo species
1,DOID:332,Disease_ChemicalEntity,6591.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,"1,1,2,2-tetrachloroethane","1,1,2,2-tetrachloroethane",C(C(Cl)Cl)(Cl)Cl,C(C(Cl)Cl)(Cl)Cl,amyotrophic lateral sclerosis,Generalised,Homo species
2,DOID:4790,Disease_ChemicalEntity,7845.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,"buta-1,3-diene","1,3-butadiene",C=CC=C,C=CC=C,medulloepithelioma,Generalised,Homo species
3,DOID:7398,Disease_ChemicalEntity,7845.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,"buta-1,3-diene","1,3-butadiene",C=CC=C,C=CC=C,cerebral primitive neuroectodermal tumor,Generalised,Homo species
4,DOID:0050902,Disease_ChemicalEntity,7845.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,"buta-1,3-diene","1,3-butadiene",C=CC=C,C=CC=C,medulloblastoma,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1372,DOID:10283,Disease_ChemicalEntity,23994.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,zinc,Zinc,[Zn],[Zn],prostate cancer,Generalised,Homo species
1373,DOID:11165,Disease_ChemicalEntity,23994.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,zinc,Zinc,[Zn],[Zn],common wart,Generalised,Homo species
1374,DOID:9352,Disease_ChemicalEntity,23994.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,zinc,Zinc,[Zn],[Zn],type 2 diabetes mellitus,Generalised,Homo species
1375,DOID:10603,Disease_ChemicalEntity,23994.0,Disease,linked to,ChemicalEntity,MonarchKG,DOID,Pubchem,zinc,Zinc,[Zn],[Zn],glucose intolerance,Generalised,Homo species


In [11]:
PrimeKG_Disease_Chemical2

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_do_name,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,head_detail_name,kg_type,species
0,DOID:10763,Disease_ChemicalEntity,3278,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,NaN,"2-[2,3-dichloro-4-(2-methylidenebutanoyl)pheno...",Etacrynic acid,CCC(=C)C(=O)C1=C(C(=C(C=C1)OCC(=O)O)Cl)Cl,CCC(=C)C(=O)C1=C(C(=C(C=C1)OCC(=O)O)Cl)Cl,hypertension,Generalised,Homo species
1,DOID:10763,Disease_ChemicalEntity,2471,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,NaN,3-(butylamino)-4-phenoxy-5-sulfamoylbenzoic acid,Bumetanide,CCCCNC1=C(C(=CC(=C1)C(=O)O)S(=O)(=O)N)OC2=CC=C...,CCCCNC1=C(C(=CC(=C1)C(=O)O)S(=O)(=O)N)OC2=CC=C...,hypertension,Generalised,Homo species
2,DOID:0050425,Disease_ChemicalEntity,6047,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,MONDO:0005391,"(2S)-2-amino-3-(3,4-dihydroxyphenyl)propanoic ...",Levodopa,C1=CC(=C(C=C1C[C@@H](C(=O)O)N)O)O,C1=CC(=C(C=C1C[C@@H](C(=O)O)N)O)O,restless legs syndrome,Generalised,Homo species
3,DOID:0050425,Disease_ChemicalEntity,34359,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,MONDO:0005391,"(2S)-3-(3,4-dihydroxyphenyl)-2-hydrazinyl-2-me...",Carbidopa,C[C@](CC1=CC(=C(C=C1)O)O)(C(=O)O)NN,C[C@](CC1=CC(=C(C=C1)O)O)(C(=O)O)NN,restless legs syndrome,Generalised,Homo species
4,DOID:4752,Disease_ChemicalEntity,5745,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,MONDO:0007803,"[2-[(8S,9S,10R,13S,14S,17R)-17-hydroxy-10,13-d...",Cortisone acetate,CC(=O)OCC(=O)[C@]1(CC[C@@H]2[C@@]1(CC(=O)[C@H]...,CC(=O)OCC(=O)[C@]1(CC[C@@H]2[C@@]1(CC(=O)[C@H]...,multiple system atrophy,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1364,DOID:11405,Disease_ChemicalEntity,14969,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,MONDO:0005504,"(1S,2R,18R,19R,22S,25R,28R,40S)-48-[(2S,3R,4S,...",Vancomycin,C[C@H]1[C@H]([C@@](C[C@@H](O1)O[C@@H]2[C@H]([C...,C[C@H]1[C@H]([C@@](C[C@@H](O1)O[C@@H]2[C@H]([C...,diphtheria,Generalised,Homo species
1365,DOID:2730,Disease_ChemicalEntity,446596,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,MONDO:0006541,"9-[(E)-4-[(2S,3R,4R,5S)-3,4-dihydroxy-5-[[(2S,...",Mupirocin,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,epidermolysis bullosa,Generalised,Homo species
1366,DOID:4644,Disease_ChemicalEntity,446596,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,MONDO:0017610,"9-[(E)-4-[(2S,3R,4R,5S)-3,4-dihydroxy-5-[[(2S,...",Mupirocin,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,epidermolysis bullosa simplex,Generalised,Homo species
1367,DOID:11907,Disease_ChemicalEntity,446596,Disease,off-label use,ChemicalEntity,MonarchKG,DOID,Pubchem,MONDO:0001404,"9-[(E)-4-[(2S,3R,4R,5S)-3,4-dihydroxy-5-[[(2S,...",Mupirocin,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,ecthyma,Generalised,Homo species


In [12]:
PrimeKG_Disease_Chemical3

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,tail_detail_name,tail_pubchem_id,tail_smiles_name,tail_smiles_name.1,head_detail_name,kg_type,species
0,DOID:10763,Disease_ChemicalEntity,9601226,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,"(2S,4S)-4-cyclohexyl-1-[2-[[(1S)-2-methyl-1-pr...",Fosinopril,CCC(=O)O[C@H](C(C)C)O[P@](=O)(CCCCC1=CC=CC=C1)...,CCC(=O)O[C@H](C(C)C)O[P@](=O)(CCCCC1=CC=CC=C1)...,hypertension,Generalised,Homo species
1,DOID:10763,Disease_ChemicalEntity,5464343,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,(4S)-3-[(2S)-2-[[(2S)-1-ethoxy-1-oxo-4-phenylb...,Imidapril,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@@H](C)C(=O)N2[...,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@@H](C)C(=O)N2[...,hypertension,Generalised,Homo species
2,DOID:10763,Disease_ChemicalEntity,56330,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,"(4S,7S)-7-[[(2S)-1-ethoxy-1-oxo-4-phenylbutan-...",Cilazapril,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@H]2CCCN3CCC[C@...,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@H]2CCCN3CCC[C@...,hypertension,Generalised,Homo species
3,DOID:10763,Disease_ChemicalEntity,2162,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,3-O-ethyl 5-O-methyl 2-(2-aminoethoxymethyl)-4...,Amlodipine,CCOC(=O)C1=C(NC(=C(C1C2=CC=CC=C2Cl)C(=O)OC)C)C...,CCOC(=O)C1=C(NC(=C(C1C2=CC=CC=C2Cl)C(=O)OC)C)C...,hypertension,Generalised,Homo species
4,DOID:10763,Disease_ChemicalEntity,5484727,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,"(2S,3aR,7aS)-1-[(2S)-2-[[(2S)-1-ethoxy-1-oxo-4...",Trandolapril,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@@H](C)C(=O)N2[...,CCOC(=O)[C@H](CCC1=CC=CC=C1)N[C@@H](C)C(=O)N2[...,hypertension,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5523,DOID:12359,Disease_ChemicalEntity,DB06343,Disease,indication,ChemicalEntity,PrimeKG,DOID,DRUGBANK,Teprotumumab,Teprotumumab,NaN,NaN,endocrine exophthalmos,Generalised,Homo species
5524,DOID:314,Disease_ChemicalEntity,25151352,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,"5-[(5-chloro-1H-pyrrolo[2,3-b]pyridin-3-yl)met...",Pexidartinib,C1=CC(=NC=C1CC2=CNC3=C2C=C(C=N3)Cl)NCC4=CN=C(C...,C1=CC(=NC=C1CC2=CNC3=C2C=C(C=N3)Cl)NCC4=CN=C(C...,tenosynovial giant cell tumor,Generalised,Homo species
5525,DOID:2702,Disease_ChemicalEntity,25151352,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,"5-[(5-chloro-1H-pyrrolo[2,3-b]pyridin-3-yl)met...",Pexidartinib,C1=CC(=NC=C1CC2=CNC3=C2C=C(C=N3)Cl)NCC4=CN=C(C...,C1=CC(=NC=C1CC2=CNC3=C2C=C(C=N3)Cl)NCC4=CN=C(C...,pigmented villonodular synovitis,Generalised,Homo species
5526,DOID:2215,Disease_ChemicalEntity,DB00036,Disease,indication,ChemicalEntity,PrimeKG,DOID,DRUGBANK,Coagulation factor VIIa Recombinant Human,Coagulation factor VIIa Recombinant Human,NaN,NaN,factor VII deficiency,Generalised,Homo species


### 3c. PharmKG

In [13]:
PharmKG_Disease_Chemical = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Disease_Chemical.csv')
PharmKG_Disease_Chemical.columns = PharmKG_Disease_Chemical.columns.str.lower()
PharmKG_Disease_Chemical['head_detail_name'] = PharmKG_Disease_Chemical['head'].apply(
    lambda x: DOID_disname_dict.get(x)
              if isinstance(x, str) and x.startswith('DOID')
              else (MESH_dict.get(x) or Mesh_supp_dict.get(x))
)
PharmKG_Disease_Chemical['head_id_is'] = np.where(
    PharmKG_Disease_Chemical['head'].isna(), None,
    np.where(PharmKG_Disease_Chemical['head'].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
)
PharmKG_Disease_Chemical['tail'] = PharmKG_Disease_Chemical['tail'].astype(str)
PharmKG_Disease_Chemical['tail_id_is'] = np.where(
    PharmKG_Disease_Chemical['tail'].isna(),None,
    np.where(PharmKG_Disease_Chemical['tail'].str.startswith('DB'), 'DRUGBANK', 'Pubchem')
)
print(f"PharmKG:    {len(PharmKG_Disease_Chemical):,} rows")

PharmKG_Disease_Chemical['kg_type'] = 'Generalised'
PharmKG_Disease_Chemical['kg_source'] = 'PharmKG'
PharmKG_Disease_Chemical['species'] = 'Homo species'

PharmKG_Disease_Chemical

PharmKG:    11 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_orignal,tail_orignal,pubmed_id,sentence_tokenized,head_detail_name,kg_type,species
0,DOID:10017,Disease_Chemical,5724,disease,J,chemical,PharmKG,DOID,Pubchem,multiple endocrine neoplasia type 1,zes,"11144036.0, 11573043.0, 11408501.0, 22902777.0...",'Twenty-two percent of the patients had multip...,multiple endocrine neoplasia type 1,Generalised,Homo species
1,DOID:10017,Disease_Chemical,168010204,disease,J,chemical,PharmKG,DOID,Pubchem,multiple endocrine neoplasia type 1,parathyroid,"11134142.0, 8917740.0, 2568587.0, 23565397.0, ...",'Multiple_endocrine_neoplasia_type_1 -LRB- MEN...,multiple endocrine neoplasia type 1,Generalised,Homo species
2,DOID:2377,Disease_Chemical,134693868,disease,L,chemical,PharmKG,DOID,Pubchem,multiple sclerosis,eae,"22942767.0, 11440744.0, 17943249.0, 17967467.0...",'The Immunomodulatory and Neuroprotective Effe...,multiple sclerosis,Generalised,Homo species
3,DOID:552,Disease_Chemical,71576778,disease,Sa,chemical,PharmKG,DOID,Pubchem,pneumonia,mds,NaN,NaN,pneumonia,Generalised,Homo species
4,DOID:9821,Disease_Chemical,17220,disease,V+,chemical,PharmKG,DOID,Pubchem,choroideremia,chm,"23460904.0, 10447648.0, 17935254.0, 24962736.0...",'Choroideremia -LRB- CHM -RRB- is an X-linked ...,choroideremia,Generalised,Homo species
5,DOID:2121,Disease_Chemical,20796,disease,U,chemical,PharmKG,DOID,Pubchem,ectodermal dysplasia,eds,"24047847.0, 6837633.0",'We have observed ectodermal_dysplasia -LRB- E...,ectodermal dysplasia,Generalised,Homo species
6,DOID:9821,Disease_Chemical,17220,disease,H,chemical,PharmKG,DOID,Pubchem,choroideremia,chm,"9067750.0, 28009400.0, 12827496.0, 12827496.0,...",'Choroideremia -LRB- CHM -RRB- is an X-linked_...,choroideremia,Generalised,Homo species
7,DOID:2121,Disease_Chemical,15906,disease,L,chemical,PharmKG,DOID,Pubchem,ectodermal dysplasia,hed,"26893642.0, 17181574.0, 21771270.0, 10903849.0...",'UNASSIGNED : Ectodysplasin -LRB- EDA -RRB- ge...,ectodermal dysplasia,Generalised,Homo species
8,DOID:10017,Disease_Chemical,12679,disease,Y,chemical,PharmKG,DOID,Pubchem,multiple endocrine neoplasia type 1,hpt,"26541865.0, 19650788.0, 9681845.0, 14715834.0,...",'BACKGROUND : Primary_hyperparathyroidism -LRB...,multiple endocrine neoplasia type 1,Generalised,Homo species
9,DOID:10017,Disease_Chemical,135398675,disease,V+,chemical,PharmKG,DOID,Pubchem,multiple endocrine neoplasia type 1,gsp,"8964883.0, 8964883.0","'Recently , a high frequency of gsp mutation i...",multiple endocrine neoplasia type 1,Generalised,Homo species


### 3d. TARKG

In [14]:
TARKG_Disease_Chemical = pd.read_csv(PROC_DIR + 'TARKG/Disease_Compound_TARKG.csv')
TARKG_Disease_Chemical.columns = TARKG_Disease_Chemical.columns.str.lower()
print(f"TARKG:      {len(TARKG_Disease_Chemical):,} rows")

TARKG_Disease_Chemical['kg_type'] = 'Generalised'
TARKG_Disease_Chemical['kg_source'] = 'TARKG'
TARKG_Disease_Chemical['species'] = 'Homo species'
TARKG_Disease_Chemical['relation'] = 'Disease_ChemicalEntity'
TARKG_Disease_Chemical['tail_type'] = 'ChemicalEntity'

TARKG_Disease_Chemical.head(2)

TARKG:      105 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_iupac,tail_smiles,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,DOID:2018,Disease_ChemicalEntity,474,Disease,Mp,ChemicalEntity,TARKG,hyperinsulinism,"[5-(6-aminopurin-9-yl)-3,4-dihydroxyoxolan-2-y...",CN(C)OP(=O)(O)OCC1C(C(C(O1)N2C=NC3=C(N=CN=C32)...,DOID,Pubchem,PharmKG-349334,PharmKG,1,Generalised,Homo species
1,DOID:4483,Disease_ChemicalEntity,84029,Disease,Mp,ChemicalEntity,TARKG,rhinitis,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,DOID,Pubchem,PharmKG-314512,PharmKG,1,Generalised,Homo species


### 3e Hald

In [15]:
hald = pd.read_csv(PROC_DIR + 'hald/Disease_Chemical_HALD.csv')
hald.columns = hald.columns.str.lower()
print(f"TARKG:      {len(hald):,} rows")
hald['kg_type'] = 'Aging'
# hald['kg_source'] = 'TARKG'
hald['species'] = 'Homo species'

hald.head(2)

TARKG:      2,706 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,DOID:1307,Disease_ChemicalEntity,5997,Disease,commence,ChemicalEntity,HALD_KG,Dementia,C[C@H](CCCC(C)C)[C@H]1CC[C@@H]2[C@@]1(CC[C@H]3...,"(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-[...",DOID,Pubchem,Aging,Homo species
1,DOID:552,Disease_ChemicalEntity,54670067,Disease,prophylactic use of,ChemicalEntity,HALD_KG,Pneumonia,C([C@@H]([C@@H]1C(=C(C(=O)O1)O)O)O)O,"(2R)-2-[(1S)-1,2-dihydroxyethyl]-3,4-dihydroxy...",DOID,Pubchem,Aging,Homo species


## 4. Check and Fix Duplicate Columns

In [16]:
SOURCE_DFS = [
    ('Monarch_Disease_Chemical',   Monarch_Disease_Chemical),
    ('PrimeKG_Disease_Chemical1',  PrimeKG_Disease_Chemical1),
    ('PrimeKG_Disease_Chemical2',  PrimeKG_Disease_Chemical2),
    ('PrimeKG_Disease_Chemical3',  PrimeKG_Disease_Chemical3),
    ('PharmKG_Disease_Chemical',   PharmKG_Disease_Chemical),
    ('TARKG_Disease_Chemical',     TARKG_Disease_Chemical),
    ('hald', hald),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Disease_Chemical] ✓ no duplicates
[PrimeKG_Disease_Chemical1] ✓ no duplicates
[PrimeKG_Disease_Chemical2] ✓ no duplicates
[PrimeKG_Disease_Chemical3] ✓ no duplicates
[PharmKG_Disease_Chemical] ✓ no duplicates
[TARKG_Disease_Chemical] ✓ no duplicates
[hald] ✓ no duplicates


In [17]:
# Monarch_Disease_Chemical  = Monarch_Disease_Chemical.loc[:,  ~Monarch_Disease_Chemical.columns.duplicated(keep='first')]
# PrimeKG_Disease_Chemical1 = PrimeKG_Disease_Chemical1.loc[:, ~PrimeKG_Disease_Chemical1.columns.duplicated(keep='first')]
# PrimeKG_Disease_Chemical2 = PrimeKG_Disease_Chemical2.loc[:, ~PrimeKG_Disease_Chemical2.columns.duplicated(keep='first')]
# PrimeKG_Disease_Chemical3 = PrimeKG_Disease_Chemical3.loc[:, ~PrimeKG_Disease_Chemical3.columns.duplicated(keep='first')]
# PharmKG_Disease_Chemical  = PharmKG_Disease_Chemical.loc[:,  ~PharmKG_Disease_Chemical.columns.duplicated(keep='first')]
# TARKG_Disease_Chemical    = TARKG_Disease_Chemical.loc[:,    ~TARKG_Disease_Chemical.columns.duplicated(keep='first')]

# for name, df in SOURCE_DFS:
#     remaining = df.columns[df.columns.duplicated()].tolist()
#     print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

## 5. Align Schemas and Concatenate

In [18]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 11,116 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C0568062,Disease_ChemicalEntity,126941,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,methotrexate response - Toxicity,methotrexate,Generalised,Homo species
1,DOID:670,Disease_ChemicalEntity,3007,Disease,related_to,ChemicalEntity,MonarchKG,DOID,Pubchem,amphetamine abuse,amphetamine,Generalised,Homo species


## 6. Standardise Fixed-Value Columns

In [19]:
final_df['head_type'] = 'Disease'
final_df['tail_type'] = 'ChemicalEntity'
final_df['relation']  = 'Disease_ChemicalEntity'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))

Unique relation: {'Disease_ChemicalEntity'}
Unique head_id_is: {'MESH', 'DOID'}
Unique tail_id_is: {'DRUGBANK', 'Pubchem', 'Drugbank', nan}


## 7. Deduplicate

In [20]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 9,500 rows


## 8. Resolve Tail (Chemical) Names

1. PubChem IUPAC lookup.
2. Normalise DrugBank IDs; DrugBank name fallback.
3. Re-assign `tail_id_is` and `head_id_is` after normalisation.
4. Drop unresolvable rows.

In [22]:
# Step 1 – PubChem lookup
final_df_dedup['tail_detail_name'] = final_df_dedup['tail'].astype(str).map(Pubchem_IUPAC_CID_Dict)
final_df_dedup['tail_smiles_name'] = final_df_dedup['tail'].astype(str).map(Pubchem_CID_Smile_Dict)

# Step 2 – Normalise DrugBank IDs and fallback
def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df_dedup['tail'] = final_df_dedup['tail'].apply(standardize_drugbank_id).astype(str)
mask = final_df_dedup['tail_detail_name'].isna() & final_df_dedup['tail'].str.startswith('DB')
final_df_dedup.loc[mask, 'tail_detail_name'] = final_df_dedup.loc[mask, 'tail'].map(Drugbank_dict)

# Step 3 – Re-assign id_is after normalisation
final_df_dedup['head_id_is'] = np.where(
    final_df_dedup['head'].isna(), None,
    np.where(final_df_dedup['head'].astype(str).str.startswith('DOID'), 'DOID', 'MESH_ID')
)
final_df_dedup['tail_id_is'] = np.where(
    final_df_dedup['tail'].isna(), None,
    np.where(final_df_dedup['tail'].str.startswith('DB'), 'Drugbank', 'Pubchem')
)

# Step 4 – Drop unresolvable rows
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[
    ~final_df_dedup['head_detail_name'].isna() &
    ~final_df_dedup['tail_detail_name'].isna()
].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} unresolvable rows → {len(final_df_dedup):,} remaining")

Dropped 1,561 unresolvable rows → 7,939 remaining


## 9. Add Schema Columns and Enforce Column Order

In [23]:
EXTRA_COLS = ['tail_smiles_name']
final_df_dedup = final_df_dedup[REQUIRED_COLS + EXTRA_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (7939, 14)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,tail_smiles_name
0,C0342720,Disease_ChemicalEntity,74413906,Disease,related_to,ChemicalEntity,MonarchKG,MESH_ID,Pubchem,Vitamin B12-responsive methylmalonic acidemia,"cobalt(3+);[5-(5,6-dimethylbenzimidazol-1-yl)-...",Generalised,Homo species,CC1=CC2=C(C=C1C)N(C=N2)C3C(C(C(O3)CO)OP(=O)([O...
1,C0432371,Disease_ChemicalEntity,446220,Disease,related_to,ChemicalEntity,MonarchKG,MESH_ID,Pubchem,Fetal cocaine syndrome,"methyl (1R,2R,3S,5S)-3-benzoyloxy-8-methyl-8-a...",Generalised,Homo species,CN1[C@H]2CC[C@@H]1[C@H]([C@H](C2)OC(=O)C3=CC=C...
2,C0568062,Disease_ChemicalEntity,126941,Disease,related_to,ChemicalEntity,MonarchKG,MESH_ID,Pubchem,methotrexate response - Toxicity,"(2S)-2-[[4-[(2,4-diaminopteridin-6-yl)methyl-m...",Generalised,Homo species,CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C...
3,C1334749,Disease_ChemicalEntity,126941,Disease,related_to,ChemicalEntity,MonarchKG,MESH_ID,Pubchem,Methotrexate-associated lymphoproliferative di...,"(2S)-2-[[4-[(2,4-diaminopteridin-6-yl)methyl-m...",Generalised,Homo species,CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C...
4,C1851920,Disease_ChemicalEntity,6047,Disease,related_to,ChemicalEntity,MonarchKG,MESH_ID,Pubchem,Dystonia 5,"(2S)-2-amino-3-(3,4-dihydroxyphenyl)propanoic ...",Generalised,Homo species,C1=CC(=C(C=C1C[C@@H](C(=O)O)N)O)O


## 10. QC — NaN Check

In [24]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 11. Save Output

In [25]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 7,939 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_CHEMICALENTITY/ALL_DISEASE_CHEMICALENTITY.csv


# Final Mapping of disease

In [26]:
import pandas as pd
import numpy as np
import re

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_CHEMICALENTITY/ALL_DISEASE_CHEMICALENTITY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

In [27]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [28]:
# =========================================================
# LOAD KG FILE
# =========================================================

final_file = pd.read_csv(OUT_PATH)

# =========================================================
# MAP DISEASE IDS TO TAIL
# =========================================================

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='head')

# =========================================================
# RESULTS
# =========================================================
final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,tail_smiles_name
0,C0342720,Disease_ChemicalEntity,74413906,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Vitamin B12-responsive methylmalonic acidemia,"cobalt(3+);[5-(5,6-dimethylbenzimidazol-1-yl)-...",Generalised,Homo species,CC1=CC2=C(C=C1C)N(C=N2)C3C(C(C(O3)CO)OP(=O)([O...
1,C0432371,Disease_ChemicalEntity,446220,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Fetal cocaine syndrome,"methyl (1R,2R,3S,5S)-3-benzoyloxy-8-methyl-8-a...",Generalised,Homo species,CN1[C@H]2CC[C@@H]1[C@H]([C@H](C2)OC(=O)C3=CC=C...
2,C0568062,Disease_ChemicalEntity,126941,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,methotrexate response - Toxicity,"(2S)-2-[[4-[(2,4-diaminopteridin-6-yl)methyl-m...",Generalised,Homo species,CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C...
3,C1334749,Disease_ChemicalEntity,126941,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Methotrexate-associated lymphoproliferative di...,"(2S)-2-[[4-[(2,4-diaminopteridin-6-yl)methyl-m...",Generalised,Homo species,CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C...
4,C1851920,Disease_ChemicalEntity,6047,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Dystonia 5,"(2S)-2-amino-3-(3,4-dihydroxyphenyl)propanoic ...",Generalised,Homo species,C1=CC(=C(C=C1C[C@@H](C(=O)O)N)O)O
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7934,DOID:9993,Disease_ChemicalEntity,448601,Disease,receive,ChemicalEntity,HALD_KG,DOID,Pubchem,Hypoglycemia,"(4R,7S,10S,13R,16S,19R)-10-(4-aminobutyl)-19-[...",Aging,Homo species,C[C@H]([C@H]1C(=O)N[C@@H](CSSC[C@@H](C(=O)N[C@...
7935,DOID:9993,Disease_ChemicalEntity,5280933,Disease,treated,ChemicalEntity,HALD_KG,DOID,Pubchem,Hypoglycemia,"(6Z,9Z,12Z)-octadeca-6,9,12-trienoic acid",Aging,Homo species,CCCCC/C=C\C/C=C\C/C=C\CCCCC(=O)O
7936,DOID:9993,Disease_ChemicalEntity,5793,Disease,associated,ChemicalEntity,HALD_KG,DOID,Pubchem,Hypoglycemia,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",Aging,Homo species,C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O
7937,DOID:9993,Disease_ChemicalEntity,64689,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,hypoglycemia,"(2R,3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,...",Generalised,Homo species,C([C@@H]1[C@H]([C@@H]([C@H]([C@@H](O1)O)O)O)O)O


In [33]:
final_file = final_file[~final_file['head'].isna()]
final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,tail_smiles_name
0,C0342720,Disease_ChemicalEntity,74413906,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Vitamin B12-responsive methylmalonic acidemia,"cobalt(3+);[5-(5,6-dimethylbenzimidazol-1-yl)-...",Generalised,Homo species,CC1=CC2=C(C=C1C)N(C=N2)C3C(C(C(O3)CO)OP(=O)([O...
1,C0432371,Disease_ChemicalEntity,446220,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Fetal cocaine syndrome,"methyl (1R,2R,3S,5S)-3-benzoyloxy-8-methyl-8-a...",Generalised,Homo species,CN1[C@H]2CC[C@@H]1[C@H]([C@H](C2)OC(=O)C3=CC=C...
2,C0568062,Disease_ChemicalEntity,126941,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,methotrexate response - Toxicity,"(2S)-2-[[4-[(2,4-diaminopteridin-6-yl)methyl-m...",Generalised,Homo species,CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C...
3,C1334749,Disease_ChemicalEntity,126941,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Methotrexate-associated lymphoproliferative di...,"(2S)-2-[[4-[(2,4-diaminopteridin-6-yl)methyl-m...",Generalised,Homo species,CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C...
4,C1851920,Disease_ChemicalEntity,6047,Disease,related_to,ChemicalEntity,MonarchKG,MESH,Pubchem,Dystonia 5,"(2S)-2-amino-3-(3,4-dihydroxyphenyl)propanoic ...",Generalised,Homo species,C1=CC(=C(C=C1C[C@@H](C(=O)O)N)O)O
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7934,DOID:9993,Disease_ChemicalEntity,448601,Disease,receive,ChemicalEntity,HALD_KG,DOID,Pubchem,Hypoglycemia,"(4R,7S,10S,13R,16S,19R)-10-(4-aminobutyl)-19-[...",Aging,Homo species,C[C@H]([C@H]1C(=O)N[C@@H](CSSC[C@@H](C(=O)N[C@...
7935,DOID:9993,Disease_ChemicalEntity,5280933,Disease,treated,ChemicalEntity,HALD_KG,DOID,Pubchem,Hypoglycemia,"(6Z,9Z,12Z)-octadeca-6,9,12-trienoic acid",Aging,Homo species,CCCCC/C=C\C/C=C\C/C=C\CCCCC(=O)O
7936,DOID:9993,Disease_ChemicalEntity,5793,Disease,associated,ChemicalEntity,HALD_KG,DOID,Pubchem,Hypoglycemia,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",Aging,Homo species,C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O
7937,DOID:9993,Disease_ChemicalEntity,64689,Disease,indication,ChemicalEntity,PrimeKG,DOID,Pubchem,hypoglycemia,"(2R,3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,...",Generalised,Homo species,C([C@@H]1[C@H]([C@@H]([C@H]([C@@H](O1)O)O)O)O)O


In [34]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 7,933 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_CHEMICALENTITY/ALL_DISEASE_CHEMICALENTITY.csv
